In [14]:
# imports
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [8]:
# load the cleaned data
df = pd.read_csv("telco_clean.csv")
print(df.shape)
df.head()

(7032, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [9]:
# turn text columns into numbers, then split 80/20
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5625, 30) (1407, 30)
0.2657777777777778 0.2658137882018479


In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# logistic regression (baseline), with scaling
log_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)

# decision tree
tree_model = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)

# random forest
rf_model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [11]:
# quick check that the models work
for name, pred in [("Logistic regression", log_pred),
                   ("Decision tree", tree_pred),
                   ("Random forest", rf_pred)]:
    print(name)
    print(classification_report(y_test, pred))

Logistic regression
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407

Decision tree
              precision    recall  f1-score   support

           0       0.90      0.68      0.77      1033
           1       0.47      0.78      0.59       374

    accuracy                           0.71      1407
   macro avg       0.68      0.73      0.68      1407
weighted avg       0.78      0.71      0.72      1407

Random forest
              precision    recall  f1-score   support

           0       0.87      0.81      0.84      1033
           1       0.56      0.66      0.61       374

    accuracy                           0.77      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79   

In [12]:
# save everything for the evaluation part
joblib.dump(log_model, "log_model.pkl")
joblib.dump(tree_model, "tree_model.pkl")
joblib.dump(rf_model, "rf_model.pkl")

results = X_test.copy()
results["actual"] = y_test
results["logistic_pred"] = log_pred
results["tree_pred"] = tree_pred
results["forest_pred"] = rf_pred
results.to_csv("test_predictions.csv", index=False)